# Tutorial

In [1]:
import pandas as pd
import pint
ureg = pint.get_application_registry()

In [2]:
from aircraftdetective.processing.usdot import process_data_usdot_t2
from aircraftdetective.processing.acftdb import _read_engine_database
from aircraftdetective.processing.literature import (
    process_data_weinold_database,
    process_data_babikian_figures
)

from aircraftdetective.calculations.engines import (
    determine_takeoff_to_cruise_tsfc_ratio,
    scale_engine_data_from_icao_emissions_database
)
from aircraftdetective.calculations.engines import (
    calculate_air_mass_flow_rate,
    calculate_engine_efficiencies
)
from aircraftdetective.calculations.aerodynamics import (
    compute_lift_to_drag_ratio,
    compute_aspect_ratio
)

from aircraftdetective.utility.tabular import (
    left_merge_wildcard,
    export_typed_dataframe_to_excel
)

Aircraft specifications are provided in an Excel file hosted at the Zenodo repository ["Aircraft Detective" Dataset](https://doi.org/10.5281/zenodo.14382100).  
They can be loaded using the [`aircraftdetective.processing.literature.process_data_weinold_database`](https://aircraftdetective.readthedocs.io/en/latest/api/processing/literature/#aircraftdetective.processing.literature.process_data_weinold_database) function.

In [3]:
df_acft = process_data_weinold_database()

In [4]:
df_acft.head(5)

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,Number of Engines,Sources (Weights),Comments,Payload/Range: Range at Point B,Payload/Range: Range at Point C,Payload/Range: ZFW at Point B,Payload/Range: ZFW at Point C,Payload/Range: MTOW,Source (Payload/Range),All Data hamonized?
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,2.0,https://aircraft.airbus.com/sites/g/files/jlcb...,NaN,3598.3749284019223,7158.606627935522,50350.0026357575,43169.9977927272,60781.0,https://aircraft.airbus.com/sites/g/files/jlcb...,YES
1,Airbus,A220-300,NaN,A220-300,A220-300 BD-500-1A11,Narrow,2016,PW1521G,PW1521G,6300.0,...,2.0,https://aircraft.airbus.com/sites/g/files/jlcb...,NaN,3681.3845400518944,6230.448593959429,55791.8615,49986.0186311349,67585.0,https://aircraft.airbus.com/sites/g/files/jlcb...,YES
2,Airbus,A300-600,A300-600,A300 B4-600,Airbus Industrie A300-600/R/CF/RCF,Wide,1984,CF6-80C2A5,CF6-80C2A5,6852.0,...,2.0,[JAWA:07/08],NaN,3621.6888888888893,5700.044444444445,129255.84717830301,115645.08717830303,165035.00191999998,https://www.airbus.com/sites/g/files/jlcbta136...,NaN
3,Airbus,A300-600,A300-600,A300 B4-601,Airbus Industrie A300-600/R/CF/RCF,Wide,1986,PW4158,PW4158,6852.0,...,2.0,[JAWA:07/08],NaN,3621.6888888888893,5700.044444444445,129255.84717830301,115645.08717830303,165035.00191999998,https://www.airbus.com/sites/g/files/jlcbta136...,NaN
4,Airbus,A310-200,NaN,A310-200,Airbus Industrie A310-200C/F,Wide,1983,JT9D-7R4D*,JT9D-7R4D*,6500.0,...,2.0,[JAWA:07/08],NaN,nan,nan,nan,nan,nan,https://www.airbus.com/sites/g/files/jlcbta136...,NaN


Engine specifications are provided in an Excel file hosted at the Zenodo repository ["Aircraft Detective" Dataset](https://doi.org/10.5281/zenodo.14382100).

In [5]:
dict_tsfc_scaling = determine_takeoff_to_cruise_tsfc_ratio(degree=2, plot=True)
df_engines_scaled = scale_engine_data_from_icao_emissions_database(
    scaling_polynomial=dict_tsfc_scaling['TSFC (cruise)'],
)
df_engines_scaled.drop(columns=['Final Test Date'], inplace=True)

In [6]:
df_engines = _read_engine_database()

In [7]:
df_merged = left_merge_wildcard(
    df_left=df_acft,
    df_right=df_engines_scaled,
    left_on='Engine Designation (ICAO)',
    right_on='Engine Identification',
)
df_merged = left_merge_wildcard(
    df_left=df_merged,
    df_right=df_engines,
    left_on='Engine Designation (aircraft-database.com)',
    right_on='Engine Designation',
)

In [8]:
df_merged = calculate_air_mass_flow_rate(df_merged)

In [9]:
df_t2 = process_data_usdot_t2()
df_with_dot = pd.merge(
    how='left',
    left=df_merged,
    right=df_t2,
    left_on='Aircraft Designation (US DOT Schedule T2)',
    right_on='Aircraft Designation (US DOT Schedule T2)'
)

In [10]:
df_with_dot = calculate_engine_efficiencies(df=df_with_dot)
df_with_dot = compute_aspect_ratio(df_with_dot)
df_with_dot = compute_lift_to_drag_ratio(df_with_dot, beta=0.04)

In [11]:
df_with_dot

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,Fuel/Revenue Seat Distance,Fuel Flow,Energy Use (per ASK),Airborne Efficiency,SLF,Engine Efficiency,Propulsive Efficiency,Thermal Efficiency,Aspect Ratio,L/D
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.017083725711339294,676.9628587507034,515672.97460341104,0.7947227191413238,0.8698859306786845,1.428333042570806e-06,0.8063583900473188,1.7713377329489786e-06,10.970703472840606,16.662689537570333
1,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.016662296961420034,678.6354643254606,498481.3456321422,0.8202657470331677,0.8621534401722212,1.428333042570806e-06,0.8059727805181062,1.7721852115807508e-06,10.970703472840606,16.662689537570333
2,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.017382073699310104,693.6825730359927,504925.0480981601,0.8202946900194606,0.8371356821737456,1.428333042570806e-06,0.8025202686738476,1.7798093061638236e-06,10.970703472840606,16.662689537570333
3,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.017430667793303437,682.9326164514346,505535.61132782587,0.8241262216332615,0.835811328558868,1.428333042570806e-06,0.8049837858047291,1.774362499914113e-06,10.970703472840606,16.662689537570333
4,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,0.021608873653475476,658.1251094063739,492133.4219351635,0.8329973838830038,0.6563284876185436,1.428333042570806e-06,0.8107269415694613,1.7617929901351742e-06,10.970703472840606,16.662689537570333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,0.029972581638940236,3040.34375,697088.1065455297,0.9142857142857143,0.6702457154656205,nan,nan,nan,nan,nan
39788,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,0.03133882098651356,2947.2153846153847,694896.1112461946,0.9582309582309583,0.639010144644835,nan,nan,nan,nan,nan
39789,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,0.0382766034390943,2956.5333333333333,639585.345620128,0.8823529411764706,0.4815436241610738,nan,nan,nan,nan,nan
39790,Lockheed,L-1011-100,L1011-1/100/200,L-1011 TriStar,Lockheed L100-10 Hercules,Wide,1975,NaN,NaN,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [14]:
df_with_dot['Energy Use (per ASK)'] = df_with_dot['Energy Use (per ASK)'].pint.to('MJ/km')

In [15]:
df_with_dot['Energy Use (per ASK)']

0        1.2129380385760185
1        1.1725008201230358
2        1.1876573480297197
3        1.1890934817863645
4        1.1575695778488695
                ...        
39787    1.6396528852772894
39788     1.634496992667272
39789    1.5043980057614432
39790                   nan
39791                   nan
Name: Energy Use (per ASK), Length: 39792, dtype: pint[megajoule / kilometer][Float64]

In [24]:
from aircraftdetective.calculations.weight import calculate_weight_metrics

In [26]:
df_with_dot = calculate_weight_metrics(df_with_dot)

In [21]:
from aircraftdetective.utility.tabular import update_column_data

In [17]:
df_literature = process_data_weinold_database(sheet_name='Literature Data')

In [19]:
df_literature.dtypes

Aircraft Designation (Literature)                                       object
L/D                                               pint[dimensionless][float64]
Source (L/D)                                                            object
Energy Use (per ASK)                      pint[megajoule / kilometer][float64]
Source (Energy Use)                                                     object
TSFC (cruise)                        pint[gram / kilonewton / second][float64]
Source (TSFC (Cruise))                                                  object
OEW/MTOW                                          pint[dimensionless][float64]
Source (OEW/MTOW)                                                       object
dtype: object

In [18]:
df_literature

,Aircraft Designation (Literature),L/D,Source (L/D),Energy Use (per ASK),Source (Energy Use),TSFC (cruise),Source (TSFC (Cruise)),OEW/MTOW,Source (OEW/MTOW)
0,A300-600,14.6026,https://doi.org/10.5281/zenodo.14560914,1.3176,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.52078,https://doi.org/10.5281/zenodo.14560914
1,A310-300,16.499,https://doi.org/10.5281/zenodo.14560914,1.4459,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.50967,https://doi.org/10.5281/zenodo.14560914
2,A320-100/200,15.9818,https://doi.org/10.5281/zenodo.14560914,1.152,https://doi.org/10.5281/zenodo.14560914,16.8925,https://doi.org/10.5281/zenodo.14560914,0.55411,https://doi.org/10.5281/zenodo.14560914
3,A380-800,nan,NaN,1.375286583998318,Rohrer et al.,nan,NaN,nan,NaN
4,B707-300,15.8095,https://doi.org/10.5281/zenodo.14560914,3.0372,https://doi.org/10.5281/zenodo.14560914,28.1765,https://doi.org/10.5281/zenodo.14560914,28.1765,https://doi.org/10.5281/zenodo.14560914
5,B720-000,15.8095,https://doi.org/10.5281/zenodo.14560914,3.0372,https://doi.org/10.5281/zenodo.14560914,nan,NaN,27.0278,https://doi.org/10.5281/zenodo.14560914
6,B720-000,nan,NaN,3.2365,https://doi.org/10.5281/zenodo.14560914,nan,NaN,nan,NaN
7,B727-200/231A,13.798,https://doi.org/10.5281/zenodo.14560914,2.0912,https://doi.org/10.5281/zenodo.14560914,23.244,https://doi.org/10.5281/zenodo.14560914,0.57158,https://doi.org/10.5281/zenodo.14560914
8,B737-100/200,13.9129,https://doi.org/10.5281/zenodo.14560914,1.9257,https://doi.org/10.5281/zenodo.14560914,22.771,https://doi.org/10.5281/zenodo.14560914,0.51984,https://doi.org/10.5281/zenodo.14560914
9,B737-300,14.0854,https://doi.org/10.5281/zenodo.14560914,1.3885,https://doi.org/10.5281/zenodo.14560914,20.8791,https://doi.org/10.5281/zenodo.14560914,0.54777,https://doi.org/10.5281/zenodo.14560914


In [27]:
update_column_data(
    df_main=df_with_dot,
    df_other=df_literature,
    merge_column='Aircraft Designation (Literature)',
    list_columns=['L/D', 'Energy Use (per ASK)', 'TSFC (cruise)', 'OEW/MTOW']
)

,Manufacturer,Aircraft Designation,Aircraft Designation (Literature),Aircraft Designation (aircraft-database.com),Aircraft Designation (US DOT Schedule T2),Type,YOI,Engine Designation (ICAO),Engine Designation (aircraft-database.com),Design Range,...,Energy Use (per ASK),Airborne Efficiency,SLF,Engine Efficiency,Propulsive Efficiency,Thermal Efficiency,Aspect Ratio,L/D,OEW/MTOW,OEW/Exit Limit
0,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,1.2129380385760185,0.7947227191413238,0.8698859306786845,1.428333042570806e-06,0.8063583900473188,1.7713377329489786e-06,10.970703472840606,16.662689537570333,0.558161648177496,293.5
1,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,1.1725008201230358,0.8202657470331677,0.8621534401722212,1.428333042570806e-06,0.8059727805181062,1.7721852115807508e-06,10.970703472840606,16.662689537570333,0.558161648177496,293.5
2,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,1.1876573480297197,0.8202946900194606,0.8371356821737456,1.428333042570806e-06,0.8025202686738476,1.7798093061638236e-06,10.970703472840606,16.662689537570333,0.558161648177496,293.5
3,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,1.1890934817863645,0.8241262216332615,0.835811328558868,1.428333042570806e-06,0.8049837858047291,1.774362499914113e-06,10.970703472840606,16.662689537570333,0.558161648177496,293.5
4,Airbus,A220-100,NaN,A220-100,A200-100 BD-500-1A10,Narrow,2016,PW1519G,PW1519G,6700.0,...,1.1575695778488695,0.8329973838830038,0.6563284876185436,1.428333042570806e-06,0.8107269415694613,1.7617929901351742e-06,10.970703472840606,16.662689537570333,0.558161648177496,293.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39973,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,1.8074,0.9142857142857143,0.6702457154656205,nan,nan,nan,nan,13.9129,0.47793,nan
39974,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,1.8074,0.9582309582309583,0.639010144644835,nan,nan,nan,nan,13.9129,0.47793,nan
39975,McDonnell Douglas,DC-10-40,DC10-40,DC-10,McDonnell Douglas DC-10-40,Wide,1973,NaN,NaN,nan,...,1.8074,0.8823529411764706,0.4815436241610738,nan,nan,nan,nan,13.9129,0.47793,nan
39976,Lockheed,L-1011-100,L1011-1/100/200,L-1011 TriStar,Lockheed L100-10 Hercules,Wide,1975,NaN,NaN,nan,...,1.6453,nan,nan,nan,nan,nan,nan,14.3153,0.53665,nan


In [24]:
df_with_dot.columns

Index(['Manufacturer', 'Aircraft Designation',
       'Aircraft Designation (Literature)',
       'Aircraft Designation (aircraft-database.com)',
       'Aircraft Designation (US DOT Schedule T2)', 'Type', 'YOI',
       'Engine Designation (ICAO)',
       'Engine Designation (aircraft-database.com)', 'Design Range',
       'Sources (Design Range)', 'Cruise Speed', 'Sources (Cruise Speed)',
       'Wingspan', 'Wing Area', 'Sources (Wing Dimensions)', 'Pax Exit Limit',
       'Average Pax', 'Sources (Pax)', 'OEW', 'MTOW', 'MZFW',
       'Number of Engines', 'Sources (Weights)', 'Comments', 'EU', 'TSFC',
       'L/D', 'Payload/Range: Range at Point B',
       'Payload/Range: Range at Point C', 'Payload/Range: ZFW at Point B',
       'Payload/Range: ZFW at Point C', 'Payload/Range: MTOW',
       'Source (Payload/Range)', 'All Data hamonized?', 'Fuel Flow (takeoff)',
       'Fuel Flow (climbout)', 'Fuel Flow (approach)', 'Fuel Flow (idle)',
       'B/P Ratio', 'Pressure Ratio', 'Rated Thrus

In [25]:
df_t2

,Year,Aircraft Designation (US DOT Schedule T2),Fuel/Available Seat Distance,Fuel/Revenue Seat Distance,Fuel Flow,Airborne Efficiency,SLF
0,1991,Boeing 757-200,0.01329377983359951,0.022279264205314737,1079.3313577391493,0.8493508088032168,0.5966884593265996
1,1991,McDonnell Douglas DC-9-10,0.03305182276276791,0.06488049041833445,929.3542842646033,0.7766995073891626,0.5094262165661414
2,1991,McDonnell Douglas DC-9-30,0.024863268467927784,0.047210231649932245,953.8429051869099,0.8001448975367419,0.5266499993537623
3,1991,McDonnell Douglas DC-9-40,0.025566983164445865,0.04696680102601686,950.8338278931751,0.7929411764705883,0.5443628819915424
4,1991,McDonnell Douglas DC-9-50,0.02415604775099102,0.04457317391750557,1110.591387226129,0.7821719169053138,0.5419413882371976
...,...,...,...,...,...,...,...
21273,2025,Airbus Industrie A320-100/200,0.012336433449789744,0.01587155588407961,974.4776785714286,0.797153024911032,0.777266799795232
21274,2025,Airbus Industrie A321/Lr,0.009550854362561301,0.01406695223141465,967.7333333333333,0.8477435681147195,0.6789569059054674
21275,2025,Airbus Industrie A321-200n,0.0072610377431749215,0.010733680523079494,791.6865191146882,0.8443764865783214,0.6764723179120418
21276,2025,Airbus Industrie A320-200n,0.009660906939498351,0.01309041806366368,781.1391349661282,0.8078299305409388,0.7380136289394035
